# 📗 부록 — 평가셋을 직접 만들기

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

> **이 노트북은 수업 시간에 다루지 않는 참고 자료입니다.** 완주 기준에 들어가지 않습니다. 관심이 있을 때 읽어 보세요.

앞 시간에 우리는 **이미 만들어져 있는 평가셋**(`guide_eval_chunk.csv`)으로 검색을 쟀습니다. 그 파일에는 질문 20개와 각 질문의 정답 조각 id 가 적혀 있었지요. 그런데 그 라벨은 **누가, 어떻게** 붙인 걸까요?

실무에서 평가셋은 대개 **직접 만들어야 합니다.** 우리 회사 문서에 맞는 평가셋을 파는 곳은 없기 때문입니다. 이 부록은 그 과정을 보여 줍니다 — 후보를 좁히고, 모델에게 판정을 맡기되 **원문 인용을 강제**하고, 그 인용이 정말 조각 안에 있는지 **기계로 검증**하고, 마지막에 **사람이 확정**합니다.

그리고 만든 평가셋을 쓰기 전에 **점검**합니다. 재는 도구가 틀리면 그 위에 쌓은 모든 수치가 함께 틀리는데, 숫자는 아무렇지 않게 나오기 때문에 알아채기가 어렵습니다.

## ⏪ 복습 — 앞 시간에 쓴 것

| 앞 시간에 한 일 | 쓴 부품 | 이 부록에서는 |
|---|---|---|
| 문서를 조각으로 자르기 | 문단을 모아 400자 근처에서 끊는 규칙 | 그대로 쓴다 — **정답 라벨이 이 규칙에 매여 있다** |
| 조각을 임베딩해 색인 | `HuggingFaceEmbeddings` + `Chroma` | 그대로 쓴다 |
| 이미 있는 평가셋으로 재기 | `hit_at_k` 등 네 지표 | 이 부록에서는 다루지 않는다 |
| 판정을 스키마로 받기 | `with_structured_output` | **라벨 판정에 그대로 쓴다** |

자르기·색인·검색·구조화 출력은 이미 배운 것입니다. 여기서 새로 쓰는 코드는 **라벨링과 점검** 부분입니다.

### 📚 공식 문서 — 오늘 배우는 것들

| 오늘 다루는 것 | 공식 문서 |
|---|---|
| 검색 파이프라인 전체(자르기·임베딩·색인·검색) | [Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval) |
| 벡터 저장소 `Chroma` | [Chroma 연동](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma) |
| 한국어 문장 임베딩 | [HuggingFace 임베딩 연동](https://docs.langchain.com/oss/python/integrations/text_embedding/huggingfacehub) |
| RAG 구조 되짚기 | [RAG](https://docs.langchain.com/oss/python/langchain/rag) |
| 판정을 스키마로 받기(`with_structured_output`) | [Models](https://docs.langchain.com/oss/python/langchain/models) |
| 스키마 필드 설명(`Field(description=...)`) | [pydantic Fields](https://docs.pydantic.dev/latest/concepts/fields/) |
| 클래스·인자 사전 | [langchain-core 레퍼런스](https://reference.langchain.com/python/langchain-core/) |

> 두 사이트의 역할이 다릅니다. `docs.langchain.com` 은 **설명과 예제**가 있는 가이드이고, `reference.langchain.com` 은 클래스와 인자를 찾아보는 **사전**입니다. 처음 배울 때는 가이드를, 인자 이름이 헷갈릴 때는 사전을 보세요.

**이 부록에서 하는 것**

- [ ] **구조화된 출력**으로 라벨 판정을 받고, 그 판정에 **원문 인용을 강제**한다
- [ ] 인용한 문장이 정말 그 조각 안에 있는지 **부분문자열로 검증**한다
- [ ] 기계가 통과시킨 라벨을 **사람이 다시 읽고** 확정한다
- [ ] 평가셋을 쓰기 전에 **일곱 항목**으로 점검한다

아래 준비 셀을 먼저 실행하세요(지난 시간과 같은 `.env` 의 `OPENAI_API_KEY` 를 씁니다).

> 이 노트북은 모델을 **열네 번** 부릅니다(질문 생성 1회 + 라벨 판정 13회). 따라하기까지 하면 스무 번입니다. 어느 절에서 몇 번 부르는지는 그 자리에 적어 두었습니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 이번 시간 공통 부품 — 앞 시간에 배운 모델

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

### 다루는 자료 — 개인정보보호위원회 안내서

실제로 배포되고 있는 **공공 안내서 두 건**을 쪽 단위로 정리한 `guide_docs.csv` 입니다. 한 행이 안내서의 **한 쪽**이고, 지어낸 문서가 아닙니다.

| 원본 | 발간 | 쪽 |
|---|---|---|
| 개인정보보호위원회 「생성형 인공지능(AI) 개발·활용을 위한 개인정보 처리 안내서」 | 2025. 8. | 52 |
| 개인정보보호위원회 「개인정보 처리방침 작성지침(표준안)」 | 2026. 2. | 43 |

> **출처 표시**: 두 안내서 모두 본문에서 *"무단전재를 금하며, 가공·인용할 때는 출처를 밝혀 주시기 바랍니다"* 라고 이용 조건을 밝히고 있습니다. 그래서 이 실습은 원본을 그대로 옮기지 않고 **파싱해 가공한 뒤 출처를 밝힙니다** — CSV 의 모든 행에 문서명과 쪽 번호가 붙어 있습니다. 자세한 내용은 `data/출처_평가코퍼스.md` 에 있습니다. **수업 밖으로 이 자료를 옮길 때도 같은 조건을 지켜 주세요.**

이 자료를 고른 이유는 분명합니다. 회사의 개인정보 담당자가 "우리 챗봇에 고객 대화를 학습에 써도 되나" 를 확인할 때 실제로 여는 문서이고, 40~50쪽이라 **"답이 어느 조각에 있나"** 가 지어낸 설정이 아니라 진짜 문제가 됩니다.

먼저 재는 **대상**인 코퍼스부터 훑어봅니다. 어떤 문서가 몇 쪽씩 들어와 있는지 봅니다.

In [ ]:
# 코퍼스를 먼저 훑어봅니다 — 한 행이 안내서의 '한 쪽' 입니다.
import pandas as pd

docs = pd.read_csv('data/guide_docs.csv')
print('문서(쪽) 수:', len(docs))

# 본문은 한 쪽이 통째로 들어 있어 표에 넣으면 읽을 수 없습니다 -- 꼬리표 열만 먼저 봅니다.
display(docs[['id', '문서', '발간', '쪽', '소제목']].head(3))

In [ ]:
# 본문은 따로 봅니다 — 이 글자들이 조각으로 잘려 검색 대상이 됩니다.
print('[', docs.loc[0, 'id'], ']', docs.loc[0, '소제목'])
print(docs.loc[0, '본문'][:250])

In [ ]:
# 두 안내서에서 각각 몇 쪽씩 들어왔는지 셉니다.
#  한쪽 안내서만 잔뜩 들어와 있으면 평가 점수가 그 문서의 성질만 반영하게 됩니다.
print(docs['문서'].value_counts().to_string())
print('한 쪽의 본문 글자 수 평균:', int(docs['본문'].str.len().mean()))

이번에는 재는 **도구**인 평가셋을 실물로 봅니다. 위 표에서 말한 세 가지가 열로 들어 있습니다 — `query`(질문) · `gold_chunks`(정답 라벨) · `근거문장`.

In [ ]:
# 평가셋 20문항을 불러옵니다 — 이 파일이 오늘 쓰는 '문제집' 입니다.
evalset = pd.read_csv('data/guide_eval_chunk.csv')
print(f'{len(evalset)}문항')

# 한 문항의 정답이 여럿일 수 있어 id 는 '|', 근거 문장은 ' || ' 로 이어 붙여 두었습니다.
display(evalset[['query_id', 'query', 'gold_chunks', '유형']].head(3))

In [ ]:
# 한 문항을 통째로 펼쳐 세 요소를 눈으로 확인합니다.
first_row = evalset.iloc[0]
print('질문      :', first_row['query'])
print('정답 라벨 :', first_row['gold_chunks'].split('|'))
print('근거 문장 :', first_row['근거문장'].split(' || ')[0][:60], '...')

# 눈금 -- 20문항이면 한 문항이 몇 점을 움직이는가.
print('한 문항의 무게:', round(1 / len(evalset), 3))

---
# 1. 라벨은 왜 조각에 붙이나 — 색인부터 세운다

## 왜 중요할까요?
정답 라벨을 붙일 곳이 두 군데 있습니다. **문서 단위**("답은 안내서 33쪽에 있다")와 **조각 단위**("33쪽을 자른 조각 중 첫 번째에 있다")입니다. 문서 단위가 편해 보이지만, **검색기가 실제로 돌려주는 것은 조각**입니다. 문서로 접는 순간 무엇이 감춰지는지 우리 색인에서 직접 봅니다.

<img src="images/문서vs청크_라벨.png" width="820">

*문서로 접으면 빗나간 조각도 적중으로 보입니다.*

먼저 지난 시간의 파이프라인을 그대로 복원합니다. **자르기 → `Document` 로 감싸기 → 색인** 순서입니다.

> ⚠️ 자르는 규칙을 바꾸면 조각 id 가 전부 달라지고, 평가셋의 **정답 라벨이 어긋납니다.** 이 평가셋은 아래 규칙으로 `size=400` 으로 자른 조각에 라벨이 붙어 있습니다 — 그래서 여기서는 규칙을 그대로 씁니다.

In [ ]:
# [제공 코드] 청킹 — RAG 파이프라인을 만든 단원의 규칙 그대로입니다(문단을 모아 size 근처에서 끊습니다).
def chunk_paragraph(text, size):
    """빈 줄로 나뉜 문단을 순서대로 모아 size 근처에서 끊는다(문단 자체는 쪼개지 않는다)."""
    chunks, cur = [], ''
    for para in [p.strip() for p in text.split('\n\n') if p.strip()]:
        if cur and len(cur) + len(para) > size:
            chunks.append(cur)
            cur = para
        else:
            cur = f'{cur}\n{para}' if cur else para
    if cur:
        chunks.append(cur)
    return chunks

In [ ]:
# 74쪽을 조각으로 자릅니다. 조각 id 는 '{문서id}-{순번}' 규칙입니다(예: ai6-0, ai6-1).
chunk_ids, chunk_texts, chunk_docs = [], [], []
for row in docs.itertuples():
    for i, piece in enumerate(chunk_paragraph(row.본문, 400)):
        chunk_ids.append(f'{row.id}-{i}')
        chunk_texts.append(piece)
        chunk_docs.append(row.id)

# 조각 본문을 id 로 바로 꺼내 쓸 수 있게 사전으로 만들어 둡니다(라벨을 눈으로 검수할 때 씁니다).
chunk_text = dict(zip(chunk_ids, chunk_texts))

print(f'문서 {len(docs)}개 -> 조각 {len(chunk_ids)}개')
print('앞 다섯 개 id:', chunk_ids[:5])

In [ ]:
# 조각을 LangChain 의 Document 로 감쌉니다 — page_content 는 검색 대상 본문, metadata 는 함께 붙일 꼬리표입니다.
#  평가에서는 검색 결과의 '본문' 이 아니라 'id' 가 필요하므로 chunk_id 를 꼬리표에 반드시 넣습니다.
from langchain_core.documents import Document

documents = [Document(page_content=text, metadata={'chunk_id': cid, 'doc_id': did})
             for cid, text, did in zip(chunk_ids, chunk_texts, chunk_docs)]
print('Document 수:', len(documents))
print('첫 조각의 꼬리표:', documents[0].metadata)

In [ ]:
# 조각을 임베딩해 색인합니다 — 지난 시간에 쓴 그 부품 그대로입니다.
#  모델을 내려받고 245개를 임베딩하느라 처음 한 번은 잠시 걸립니다.
#  색인은 이 노트북에서 가장 오래 걸리는 일이라 '한 번만' 만들고 끝까지 재사용합니다.
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')   # 임베딩 단원에서 쓴 그 한국어 모델

# ids= 를 주면 벡터DB 안에서도 우리가 정한 조각 id 가 그대로 열쇠가 됩니다.
#  ids 를 함께 넘기면 같은 id 는 덮어쓰기가 됩니다 -> 이 셀을 여러 번 실행해도 문서가 중복되지 않습니다.
store_400 = Chroma.from_documents(documents, embeddings,
                                  collection_name='guide_400', ids=chunk_ids)
print('색인 완료 — 조각', len(chunk_ids), '개')

이제 검색기를 만듭니다. 지난 시간에는 검색 결과의 **본문**을 프롬프트에 넣었지만, 평가에서는 **id** 가 필요합니다. 정답 라벨이 조각 id 이기 때문입니다. 그래서 검색 결과에서 `metadata['chunk_id']` 만 뽑는 함수를 하나 만들어 둡니다.

함수가 **색인을 인자로 받는다**는 점을 눈여겨보세요. 색인을 바꿔 가며 재야 할 때 **재는 방법은 한 글자도 바꾸지 않고 색인만 갈아 끼우기** 위해서입니다.

In [ ]:
def search_ids(store, query, k):
    """질문과 가장 가까운 조각 k개의 id 를 순위 순서로 돌려준다."""
    # k 를 그때그때 바꿔 가며 재야 하므로 검색기는 이 자리에서 만든다(색인을 감쌀 뿐이라 비용이 없다).
    retriever = store.as_retriever(search_kwargs={'k': k})
    return [d.metadata['chunk_id'] for d in retriever.invoke(query)]


# 평가셋 첫 문항으로 시험해 봅니다 -- 정답과 나란히 놓고 보면 무엇을 재려는지가 분명해집니다.
print('질문   :', first_row['query'])
print('검색 5 :', search_ids(store_400, first_row['query'], 5))
print('정답   :', first_row['gold_chunks'].split('|'))

## 문서로 접으면 무엇이 감춰지나

다른 질문 하나로 **조각 단위**와 **문서 단위**를 나란히 봅니다.

In [ ]:
# 조각 단위 -- 검색이 올린 상위 3개 조각입니다.
credit_query = '사진이나 음성 파일에서 개인정보를 지울 때 규칙과 정규표현식만으로 충분한가요?'
top3 = search_ids(store_400, credit_query, 3)
for rank, chunk_id in enumerate(top3, 1):
    print(f'{rank}위 {chunk_id}')

In [ ]:
# 같은 결과를 '문서 단위' 로 접어 봅니다 -- 조각 id 의 앞부분이 문서 id 입니다(metadata['doc_id'] 와 같은 값).
doc_rank = []
for chunk_id in top3:
    doc_id = chunk_id.split('-')[0]
    # 같은 문서에서 온 조각이 여러 개여도 문서 순위에는 한 번만 넣습니다.
    if doc_id not in doc_rank:
        doc_rank.append(doc_id)

print('문서 순위:', doc_rank)
print('이 질문의 정답 조각은 ai33-0 -- 그 조각이 속한 문서는 ai33 입니다.')
print('문서 단위로 맞혔나?', 'ai33' in doc_rank)
print('조각 단위로 맞혔나?', 'ai33-0' in top3)

In [ ]:
# 그런데 상위에 올라온 그 문서의 조각 본문을 실제로 읽어 봅니다.
print('[검색이 올린 조각]')
print(chunk_text['ai33-2'][:180])

In [ ]:
# 답이 실제로 들어 있는 조각은 따로 있습니다.
print('[답이 있는 조각]')
# 답이 되는 문장이 조각 끝에 있어서 뒤쪽 200자를 봅니다.
print(chunk_text['ai33-0'][-200:])

**문서로는 적중, 조각으로는 빗나감.**

`ai33-2` 는 노출된 개인정보의 삭제·차단 같은 다른 이야기를 하고 있어서 "규칙·정규표현식만으로 충분한가" 에 답하지 않습니다. 답은 `ai33-0` 의 마지막 문장 — *"규칙, 정규표현식 등을 통한 개인정보 검출 및 마스킹은 정확도 측면에서 한계가 있을 수 있으며, 이를 보완하기 위해 LLM 모델을 통해 …"* — 에 있고, 그 조각은 상위 3개에 올라오지 않았습니다.

문서 단위로 재면 이 문항은 **맞힌 것으로 셉니다.** 답이 없는 조각을 보여 주고도 점수를 받습니다. 그리고 답변을 만들 때 모델에게 전달되는 것은 **조각**입니다. 모델은 답할 근거가 없는 글을 받고도 무언가를 써 내려갑니다. 즉 **점수는 올라가는데 서비스는 실패하는** 상태입니다.

> **그래서 이 단원의 라벨은 조각 단위입니다.** 재는 단위를 검색기가 돌려주는 단위와 맞춥니다.

조각 단위에도 약점이 있습니다. 자르는 방법을 바꾸면 조각 id 가 전부 달라진다는 것입니다. 이 약점은 **근거 문장**으로 막습니다. "답은 `ai33-0` 에 있다" 대신 "답은 *'규칙, 정규표현식 등을 통한 …'* 이라는 문장에 있다" 로 적어 두는 것입니다. 그러면 색인을 다시 만들었을 때 그 문장을 품은 조각을 찾아 라벨을 **다시 붙일 수 있습니다.** 근거 문장을 함께 적어 두는 가장 큰 이유입니다.

## 라벨과 색인이 맞물려 있는가

라벨은 조각 id 로 되어 있습니다. 그러니 **그 id 가 지금 색인에 실제로 있어야** 채점이 성립합니다. 없는 id 를 정답이라고 적어 두면 그 문항은 영원히 0점이 되는데, **에러는 나지 않습니다.** 조용히 틀립니다. 그래서 재기 전에 한 번 확인합니다.

In [ ]:
# 평가셋의 정답 id 가 전부 색인에 있는지 확인합니다.
#  색인에 든 id 를 집합으로 한 번에 모아 둡니다(문항마다 조회하면 느립니다).
indexed_ids = set(chunk_ids)

gold_all = [c for row in evalset.itertuples() for c in row.gold_chunks.split('|')]
missing = sorted({c for c in gold_all if c not in indexed_ids})

print('정답 라벨 총 개수 :', len(gold_all))
print('색인에 없는 id    :', missing)   # 비어 있어야 정상 -- 하나라도 있으면 그 문항은 조용히 0점이 된다
print('라벨과 색인이 맞물려 있나?', len(missing) == 0)

### 🖐️ 함께 따라하기 — 다른 안내서에서 문서 단위로 접어 보기

이번에는 **다른 안내서**(처리방침 표준안, `pp` 로 시작하는 문서)에서 같은 확인을 해 봅니다.

질문: **"만 14세 미만 아동의 개인정보를 처리할 때 법정대리인 동의는 어떻게 확인하나요?"**

1. 이 질문으로 상위 **5개** 조각의 id 를 찾으세요.
2. **1위 조각의 본문 앞 200자**를 출력해 답이 실제로 들어 있는지 눈으로 확인하세요.
3. 상위 5개를 문서 단위로 접으면 문서가 **몇 개**로 줄어드는지 세어 출력하세요.

확인할 것: 조각 5개가 문서 5개가 아닙니다 — 한 문서가 여러 조각으로 상위를 차지합니다. 이것이 바로 문서 단위로 접을 때 정보가 뭉개지는 이유입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
fold_query = '만 14세 미만 아동의 개인정보를 처리할 때 법정대리인 동의는 어떻게 확인하나요?'
# 1) search_ids 로 상위 5개 조각 id 찾기 (색인은 store_400)
# 2) chunk_text 에서 1위 조각 본문 앞 200자 출력
# 3) 조각 id 앞부분(문서 id)만 모아 중복 없이 세기

### ✅ 바로 확인 퀴즈

**1.** 어떤 팀이 자기 검색기를 재 봤더니 **문서 단위 Hit@3 은 0.95, 조각 단위 Hit@3 은 0.60** 이었습니다. 이 서비스에서 실제로 무슨 일이 일어나고 있는 건가요? 한 문장으로 말해 보세요.

<details><summary>정답 보기</summary>

**맞는 문서를 찾기는 하는데 그 문서 안에서 엉뚱한 조각을 올리고 있습니다.** 두 값의 차이(0.35)가 그 비율입니다. 답변을 만들 때 전달되는 것은 조각이므로, 사용자에게는 근거 없는 답이 나갑니다.

</details>

**2.** 조각 크기를 바꿔 색인을 다시 만들었는데, 평가셋의 정답 id 를 그대로 두었습니다. 점수를 재면 어떤 일이 일어나고, 왜 알아채기 어려운가요?

<details><summary>정답 보기</summary>

없어진 id 를 정답으로 삼게 되므로 그 문항은 **무조건 0점**이 됩니다. 그런데 **에러가 나지 않습니다** — 검색은 정상으로 돌고 점수만 낮게 나옵니다. 그래서 "조각을 키웠더니 성능이 나빠졌다" 고 잘못 결론 내리기 쉽습니다. 재기 전에 **정답 id 가 색인에 있는지** 확인하는 이유입니다.

</details>

---
# 2. 라벨을 붙인다 — 그리고 검증한다

## 왜 필요할까요?
질문이 모였으면 각 질문의 답이 **어느 조각에** 있는지 정해야 합니다. 조각이 245개인데 질문마다 245개를 다 읽을 수는 없습니다. 실무에서는 이렇게 합니다.

| 단계 | 하는 일 | 왜 |
|---|---|---|
| 1단계 후보 좁히기 | 그 질문으로 검색해 상위 10~15개만 남긴다 | 사람이 읽을 수 있는 분량으로 줄인다 |
| 2단계 판정 | 후보마다 "이 조각만으로 답할 수 있나" 를 묻는다 | 정답 라벨이 무슨 뜻인지 한 문장으로 정해 둔다 |
| 3단계 근거 인용 | 답할 수 있다면 **그 문장을 원문 그대로** 옮겨 적게 한다 | 판정의 이유가 남는다 |
| 4단계 검증 | 인용한 문장이 조각 안에 실제로 있는지 기계로 확인 | 지어낸 근거를 걸러 낸다 |

<img src="images/라벨링_4단계.png" width="820">

*후보 좁히기 → 판정 → 근거 인용 → 검증. 검증을 통과 못 한 것은 버리지 않고 사람에게 넘깁니다.*

2·3단계는 사람이 해도 되고 모델에게 시켜도 됩니다. 문항이 수십 개면 모델이 훨씬 빠릅니다. 다만 **모델은 틀립니다.** 그래서 4단계 검증이 반드시 붙고, 마지막에 사람 눈이 한 번 더 지나갑니다.

4단계 검증에 대해 하나 더 짚습니다. 검증은 지어낸 근거만 걸러 내는 것이 아닙니다. **맞는 라벨을 떨어뜨리기도 합니다.** 표나 기호(`▲` 같은)가 많은 조각에서 모델이 기호를 빼고 인용하면 글자가 달라지고, 그러면 맞는 라벨인데도 검증을 통과하지 못합니다. 그래서 검증을 통과하지 못한 것은 바로 버리지 않고 **사람이 다시 보는 목록**으로 보냅니다.

> 후보 좁히기에는 한 가지 한계가 있습니다. **지금 검색기가 못 찾는 조각은 후보에도 못 들어옵니다.** 그래서 검색으로만 후보를 만들면 평가셋이 지금의 검색기 쪽으로 기웁니다. 실무에서는 문서를 직접 읽고 넣은 질문을 섞거나, 후보 수를 넉넉히 잡아 이 치우침을 줄입니다.

In [ ]:
# 1단계 후보 좁히기 -- 질문 하나로 상위 조각만 남깁니다.
label_query = '개인정보 처리방침을 만들지도, 공개하지도 않으면 어떤 제재를 받나요?'
# 12개 -- 사람이 읽어 낼 만하면서 정답을 빠뜨리지 않을 만큼 넉넉한 수입니다.
candidates = search_ids(store_400, label_query, 12)
print(candidates)

## 2~3단계 — 판정을 **구조화된 출력**으로 받는다

모델에게 두 가지를 한꺼번에 묻습니다. **답할 수 있는가**(참/거짓)와 **그 근거 문장**(문자열)입니다. 이런 답은 줄글로 받으면 곤란합니다. "네, 답할 수 있습니다. 근거는…" 같은 문장에서 참/거짓을 뽑아내려면 글자를 뒤져야 하고, 그 방식은 모델이 말투를 조금만 바꿔도 깨집니다.

앞서 배운 **구조화된 출력**이 정확히 이 자리를 위한 도구입니다. 받을 모양을 **스키마**로 선언하고 `model.with_structured_output(스키마)` 로 모델에 씌우면, 결과가 **파이썬 객체**로 돌아옵니다. `verdict.can_answer` 는 이미 `bool` 이라 `if` 에 바로 넣을 수 있습니다.

`Field(description=...)` 에 적는 설명은 **모델에게 가는 지시**입니다 — 여기에 "조각 원문 그대로" 라고 적어 두는 것이 4단계 검증의 통과율을 좌우합니다.

In [ ]:
# 판정 결과로 받을 '모양' 을 스키마로 선언합니다.
from pydantic import BaseModel, Field


class Judgement(BaseModel):
    can_answer: bool = Field(description='이 조각만으로 질문에 답할 수 있는가')
    evidence: str = Field(description='답이 되는 문장을 조각 원문 그대로. 없으면 빈 문자열')


# 모델에 스키마를 씌워 둡니다 -- 이 판정기는 아래에서 계속 재사용합니다.
judge_model = model.with_structured_output(Judgement)
print('판정 스키마 준비:', list(Judgement.model_fields))

In [ ]:
# 후보 하나를 실제로 판정해 봅니다(모델 호출 1회).
JUDGE_PROMPT = '''[질문]
{q}

[조각]
{t}

이 조각 안의 내용만으로 위 질문에 답할 수 있습니까?
답할 수 있다면 그 근거가 되는 문장을 조각에서 그대로 한 문장 옮겨 적으세요.'''


def judge(query, chunk_id):
    """조각 하나가 질문에 답할 수 있는지 판정하고 근거 문장을 함께 받는다."""
    prompt = JUDGE_PROMPT.format(q=query, t=chunk_text[chunk_id])
    return judge_model.invoke(prompt)


verdict = judge(label_query, candidates[0])
print('돌아온 것의 종류:', type(verdict).__name__)   # 문자열이 아니라 Judgement 객체
print('답할 수 있나 :', verdict.can_answer, type(verdict.can_answer).__name__)
print('근거 문장    :', verdict.evidence[:80])

## 4단계 — 인용한 문장이 조각 안에 실제로 있는가

모델이 "근거는 이 문장입니다" 라고 답했다고 그것이 조각에 있는 문장이라는 보장은 없습니다. 그래서 **기계로 확인**합니다. 줄바꿈·띄어쓰기만 다른 인용은 통과시켜야 하므로 양쪽에서 공백을 지우고 비교합니다.

In [ ]:
import re


def squeeze(text):
    """공백(띄어쓰기·줄바꿈)을 모두 지운다 -- 표기 차이 때문에 검증이 실패하지 않게."""
    return re.sub(r'\s+', '', text)


def quoted_in_chunk(sentence, chunk_id):
    """인용 문장이 그 조각 본문 안에 실제로 있는지 확인한다."""
    if not sentence.strip():          # 빈 문자열은 어떤 본문에도 '들어 있다' 가 되어 버린다
        return False
    return squeeze(sentence) in squeeze(chunk_text[chunk_id])


print('인용 검증          :', quoted_in_chunk(verdict.evidence, candidates[0]))
print('지어낸 문장으로 검증:', quoted_in_chunk('이 조각은 과태료 5천만원을 규정한다.', candidates[0]))

이제 후보 12개를 모두 판정해 라벨을 만듭니다. **모델 호출 12회**입니다(이 절의 호출은 시연 1회 + 후보 12회 = 13회). 판정과 검증을 **둘 다** 통과한 조각만 정답 라벨로 받습니다.

In [ ]:
# 후보 12개를 모두 판정해 라벨을 만듭니다(호출 12회).
gold, evidence, review_needed = [], [], []
for chunk_id in candidates:
    v = judge(label_query, chunk_id)
    if not v.can_answer:
        continue
    if quoted_in_chunk(v.evidence, chunk_id):
        gold.append(chunk_id)
        evidence.append(v.evidence)
    else:
        # '답할 수 있다' 는데 인용이 원문과 다른 경우 -- 버리지 않고 사람이 다시 볼 목록으로 보낸다.
        review_needed.append(chunk_id)

print(f'정답 조각 {len(gold)}개 / 사람이 다시 볼 것 {len(review_needed)}개')
for chunk_id, sentence in zip(gold, evidence):
    print(f'  {chunk_id} :: {sentence[:70]}')

In [ ]:
# 파일에 들어 있는 같은 질문의 라벨과 견줍니다.
saved = evalset[evalset['query_id'] == 'guide09'].iloc[0]
print('파일의 라벨 :', saved['gold_chunks'].split('|'))
print('방금 만든 것:', gold)

조각 수가 다를 수 있습니다. 파일을 만들 때는 **다른 모델**로 판정했기 때문입니다. 같은 절차를 밟아도 **누가 판정하느냐에 따라 라벨이 달라집니다.** 그래서 최종 라벨은 **파일로 고정해 두고**(더 이상 바꾸지 않고) 그 파일로만 점수를 잽니다. 매번 다시 판정하면 지난번 점수와 이번 점수를 견줄 수 없습니다.

이 절차로 20문항 전체에 라벨을 붙여 둔 것이 `guide_eval_chunk.csv` 입니다. 만드는 동안 실제로 있었던 일 세 가지를 적어 둡니다.

- **모델이 답을 못 찾은 문항이 있었습니다.** 후보 어느 것도 "답할 수 있음" 이 아니어서 그 문항은 평가셋에서   빠졌습니다(그래서 문항 번호가 `guide02` 부터 시작합니다). 사람이 다시 보고 살릴지 버릴지 정해야 하는 자리입니다.
- **작은 모델은 눈에 보이는 답도 놓쳤습니다.** 1절에서 본 `ai33-0`(규칙·정규표현식의 한계를 적은 조각)을   더 작은 모델은 "답할 수 없음" 으로 판정했습니다. 사람이 읽으면 바로 보이는 문장입니다.
- **반대 방향의 실수도 있었습니다.** 어떤 문항에서는 모델이 정답이 아닌 조각까지 정답으로 넣었습니다.   질문과 관련은 있지만 질문에 답하지는 않는 조각입니다. 예를 들어 무엇을 적어야 하는지 나열하기만 한   **목차 표**를 "파기 방법을 구분해 적어야 하나" 의 근거로 삼았습니다. 사람이 읽고 라벨 **4건**을 뺐습니다.

> 자동 판정은 **후보를 줄여 주는 도구**입니다. 라벨을 확정하는 것은 사람입니다. 평가셋은 다른 모든 수치의 기준점이므로, 여기가 틀리면 그다음 모든 판단이 함께 틀립니다.

### 🖐️ 함께 따라하기 — 다른 질문에 라벨 붙여 보기

다른 질문으로 라벨을 하나 만들어 봅니다. **모델 호출 5회**입니다.

질문: **"영상정보처리기기를 설치했을 때 처리방침에 적어야 하는 항목은 무엇인가요?"**

1. 상위 **5개** 조각을 후보로 좁히세요.
2. 후보마다 `judge` 로 판정하고, `quoted_in_chunk` 로 인용을 검증한 것만 남기세요.
3. 남은 조각 id 와 근거 문장을 출력하세요.

확인할 것: 정답 조각이 몇 개 나왔는지. 하나가 아닐 수 있습니다 — **답이 여러 조각에 나뉘어 있는 질문**은 실제로 흔합니다. 그리고 후보를 5개로 줄였으니 **더 아래에 있던 정답은 후보에도 못 들어왔다**는 점을 기억하세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
follow_label_q = '영상정보처리기기를 설치했을 때 처리방침에 적어야 하는 항목은 무엇인가요?'
# 1) search_ids 로 상위 5개 후보 좁히기 (색인은 store_400)
# 2) judge 로 판정하고 quoted_in_chunk 로 검증한 것만 남기기
# 3) 남은 조각 id 와 근거 문장 출력

### ✅ 바로 확인 퀴즈

**1.** 모델에게 "답할 수 있나" 만 묻지 않고 **근거 문장을 원문 그대로 옮기게** 하는 이유는?

<details><summary>정답 보기</summary>

판정의 이유가 남아 사람이 검수할 수 있고, 인용문이 조각 안에 실제로 있는지 **기계로 확인**해서 지어낸 판정을 걸러 낼 수 있기 때문입니다. 게다가 그 문장이 나중에 **색인을 다시 만들었을 때 라벨을 다시 붙이는 열쇠**가 됩니다.

</details>

**2.** 판정을 줄글이 아니라 **스키마(구조화된 출력)** 로 받으면 무엇이 좋은가요?

<details><summary>정답 보기</summary>

결과가 파이썬 객체로 돌아와 `verdict.can_answer` 가 이미 `bool` 이므로 **글자를 뒤져 참/거짓을 짐작할 필요가 없습니다.** 모델이 말투를 바꿔도 코드가 깨지지 않고, 필드 설명(`Field(description=...)`)으로 "원문 그대로" 같은 요구를 모델에게 정확히 전달할 수 있습니다.

</details>

**3.** 후보를 검색으로 좁히는 방식의 한계는 무엇인가요?

<details><summary>정답 보기</summary>

지금 검색기가 못 찾는 조각은 후보에 들어오지 못하므로, 평가셋이 **현재 검색기가 잘하는 쪽으로** 치우칠 수 있습니다. 문서를 직접 읽고 만든 질문을 섞거나 후보 수를 넉넉히 잡아 줄입니다.

</details>

---
# 3. 평가셋을 점검한다

## 왜 필요할까요?
라벨이 붙었다고 끝이 아닙니다. 평가셋에 결함이 있으면 **지표가 조용히 거짓말을 합니다** — 숫자는 멀쩡하게 나오는데 그 숫자가 아무것도 말해 주지 않습니다. 내보내기 전에 점검할 항목입니다.

| 점검 | 통과 못 하면 |
|---|---|
| 1. 정답 조각 id 가 색인에 실제로 있는가 | 채점이 조용히 0점 처리된다 |
| 2. 근거 문장이 그 조각 원문에 있는가 | 라벨의 근거가 지어낸 것이다 |
| 3. 질문이 문서를 베끼지 않았는가 | 점수가 실제보다 높게 나온다 |
| 4. 정답이 여럿인 문항이 섞여 있는가 | Recall 이 Hit 과 같아져 지표 넷 중 둘만 보게 된다 |
| 5. 지금 검색기가 못 찾는 문항이 있는가 | 전부 만점이라 좋아졌는지 나빠졌는지 알 수 없다 |
| 6. 한 질문의 정답 조각이 지나치게 많지 않은가 | K 를 뭘 줘도 Recall 이 오르지 않는다 |
| 7. 네 지표가 서로 다른 것을 말하는가 | 지표를 넷 만들어 놓고 둘만 보게 된다 |

> 표에 나오는 지표 이름(Hit@K·Precision@K·Recall@K·MRR)은 **앞 시간에 배운 그것들입니다.** 여기서는 "평가셋이 이런 모양이면 지표가 서로 겹친다" 는 것만 알아 두면 됩니다.

4번을 조금 더 설명합니다. 모든 문항의 정답이 하나뿐이면 Recall@K 는 Hit@K 와 **항상 같아집니다.** Precision@K 도 Hit@K 를 K 로 나눈 값이 됩니다. 지표를 넷 만들어 놓고 실제로는 두 가지만 보고 있는 셈입니다. 그래서 **정답이 둘 이상인 문항을 일부러 넣습니다.**

In [ ]:
# 1~2번 점검 -- 정답 조각 id 와 근거 문장이 실제로 있는지 확인합니다.
bad_id, bad_evidence = [], []
for row in evalset.itertuples():
    ids = row.gold_chunks.split('|')
    sentences = row.근거문장.split(' || ')
    if len(ids) != len(sentences):
        print(f'라벨과 근거 개수가 다릅니다: {row.query_id}')   # zip 은 짧은 쪽에서 조용히 끊는다
    for chunk_id, sentence in zip(ids, sentences):
        if chunk_id not in indexed_ids:
            bad_id.append(chunk_id)
        elif not quoted_in_chunk(sentence, chunk_id):
            bad_evidence.append(chunk_id)

print('없는 조각 id     :', bad_id)
print('원문에 없는 근거 :', bad_evidence)

In [ ]:
# 3번 점검 -- 질문이 문서를 베끼지 않았는가. 정답 조각들과의 겹침 평균을 문항마다 잽니다.
#  겹침은 '세 글자 덩어리' 기준입니다. 두 글자로 세면 한국어 조사·어미 때문에
#  정상 질문도 0.6 을 넘어 오탐이 납니다(실측).
def overlap_ratio(query, text):
    """질문의 세 글자 덩어리 중 몇 %가 문서 본문에 그대로 나오는가."""
    # 띄어쓰기만 다른 표현도 같은 것으로 세도록 공백을 모두 지웁니다.
    squeezed = re.sub(r'\s+', '', query)
    # 세 글자씩 한 칸 밀며 잘라, 중복 없이 모읍니다.
    grams = {squeezed[i:i + 3] for i in range(len(squeezed) - 2)}
    body = re.sub(r'\s+', '', text)
    return sum(1 for g in grams if g in body) / len(grams)


def query_overlap(row):
    """한 문항의 질문이 그 문항의 정답 조각들을 얼마나 베꼈는지의 평균."""
    ids = row.gold_chunks.split('|')
    return sum(overlap_ratio(row.query, chunk_text[c]) for c in ids) / len(ids)


evalset['겹침'] = [query_overlap(row) for row in evalset.itertuples()]
print(evalset['겹침'].describe()[['min', '50%', 'max']].round(3).to_string())
print(f"0.5 를 넘는 문항: {(evalset['겹침'] > 0.5).sum()}개")

In [ ]:
# 4번·6번 점검 -- 정답 개수. 하나뿐인 문항만 있어도 안 되고, 지나치게 많아도 안 됩니다.
evalset['정답수'] = evalset['gold_chunks'].str.split('|').apply(len)
print(evalset['정답수'].value_counts().sort_index().to_string())
print(f"복수 정답 문항: {(evalset['정답수'] > 1).sum()}개 / {len(evalset)}문항")

# 6번은 기준을 정해 두어야 판정할 수 있습니다. 여기서는 '정답이 8개를 넘으면 질문이 너무 넓다' 로 봅니다.
too_wide = evalset[evalset['정답수'] > 8]['query_id'].tolist()
print('정답 조각이 8개를 넘는 문항:', too_wide)

In [ ]:
# 5번 점검 -- 지금 검색기가 상위 3개에서 못 찾는 문항이 있는가.
def found_at_k(row, k):
    """상위 k개와 정답의 교집합. 비어 있으면 그 안에 정답이 하나도 없다는 뜻이다."""
    return set(search_ids(store_400, row.query, k)) & set(row.gold_chunks.split('|'))


missed = [row.query_id for row in evalset.itertuples() if not found_at_k(row, 3)]
print(f'상위 3개에서 못 찾은 문항 {len(missed)}개: {missed}')

1번부터 5번까지는 통과했습니다. 6번도 기준(8개)을 넘는 문항이 없어 통과입니다. 다만 **정답이 5~7개인 문항이 여덟 개**나 됩니다. 앞 시간에 "Recall 은 정답 개수와 함께 읽는다" 고 한 이유가 바로 이것입니다. 7번은 지표를 만든 뒤라야 확인할 수 있으므로 앞 시간에서 이미 봤습니다.

특히 5번에서 **못 찾는 문항이 남아 있다는 것**이 중요합니다. 전부 맞히는 평가셋은 만점밖에 나오지 않아서 무엇을 바꾸든 점수가 그대로입니다 — 그런 평가셋은 아무것도 알려 주지 못합니다. 검색을 고쳐 볼 때, **그때 움직일 여지가 있어야** 고친 것이 좋은지 나쁜지 판단할 수 있습니다.

> 점검을 통과했다고 좋은 평가셋인 것은 아닙니다. 이 항목들은 **명백한 결함이 없다**는 뜻일 뿐, 질문이 실제 사용자의 질문과 닮았는지는 여전히 사람이 판단할 몫입니다.

### 🖐️ 함께 따라하기 — 점검 항목을 하나 더 만들기

점검 항목을 하나 더 만들어 봅니다. **같은 질문이 두 번 들어가 있지 않은지**, 그리고 **정답 조각이 여러 문항에 중복해 쓰이고 있지는 않은지** 확인하는 항목입니다.

1. `evalset['query']` 에 중복이 있는지 세어 출력하세요.
2. 모든 문항의 정답 조각 id 를 모은 `gold_all` 에서 **두 번 이상** 쓰인 조각 id 를 찾아 출력하세요.

확인할 것: 중복 질문은 0개여야 합니다. 정답 조각이 겹치는 것은 결함이 아닙니다 — 다만 **한 조각이 여러 문항의 답이라면 그 조각만 잘 찾아도 점수가 오르므로** 알고는 있어야 합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) query 열의 중복 개수 세기 (duplicated().sum())
# 2) gold_all 에서 두 번 이상 등장하는 조각 id 찾기 (count 로 세면 된다)

### ✅ 바로 확인 퀴즈

**1.** 어떤 평가셋을 20문항 만들었더니 Hit@3 과 Recall@3 이 **소수점 아래까지 똑같이** 나왔습니다. 코드를 보지 않고도 이 평가셋에 대해 확실히 말할 수 있는 것은 무엇인가요?

<details><summary>정답 보기</summary>

**모든 문항의 정답이 하나뿐**입니다. 정답이 하나면 찾았을 때 1, 못 찾았을 때 0 이라 두 값이 같아집니다. 정답이 둘 이상인 문항이 하나라도 있고 그것을 부분적으로 찾았다면 두 값은 갈라집니다.

</details>

**2.** 평가셋의 모든 문항을 검색기가 맞히고 있습니다. 좋은 상태인가요?

<details><summary>정답 보기</summary>

아닙니다. 만점만 나오는 평가셋은 **변별력이 없어서** 무엇이 나아지고 나빠지는지 보여 주지 못합니다. 어려운 문항을 넣어 점수가 1.0 밑에 있도록 만들어야 합니다.

</details>

---
## 부록 정리

| 주제 | 한 일 | 남길 것 |
|---|---|---|
| 평가셋 3요소 | 질문 · 정답 라벨 · 근거 문장 | 근거 문장이 있어야 라벨을 나중에 검수할 수 있다 |
| 라벨링 4단계 | 후보 좁히기 → 판정 → 근거 인용 → 부분문자열 검증 | 판정은 **스키마로** 받고, 확정은 **사람이** 한다 |
| 인용 강제 | 모델이 지어낸 근거를 걸러 낸다 | 인용이 조각 안에 없으면 그 라벨은 버린다 |
| 점검 7항목 | 내보내기 전에 기계로 훑는다 | 특히 **전부 맞히는 평가셋은 쓸모가 없다** |

한 가지만 남긴다면: **재는 도구부터 검수하라**는 것입니다. 평가셋이 틀리면 그 위에 쌓은 모든 수치가 함께 틀리는데, 숫자는 아무렇지 않게 나오기 때문에 알아채기가 어렵습니다.